In [1]:
# =========================================================
# PS S6E4 - exp_20260410_037_dao_xgb_pair_te_posthoc_only
# minimum reproduction of Dao core:
# raw base + pairwise 171->accepted + pairwise multiclass TE + XGB
# + post-hoc optimization
# no pseudo-label / no formula-logit
# =========================================================

In [2]:
import os
import json
import itertools
import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score

class CFG:
    EXP_ID = "exp_20260410_037_dao_xgb_pair_te_posthoc_only"
    TARGET = "Irrigation_Need"
    ID_COL = "id"

    OOF_PATH = "/kaggle/input/datasets/mizushimatoshihiko/npy-files/oof_dao_xgb_pair_te_core_min_proba.npy"
    PRED_PATH = "/kaggle/input/datasets/mizushimatoshihiko/npy-files/pred_dao_xgb_pair_te_core_min_proba.npy"
    TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e4/train.csv"
    TEST_PATH = "/kaggle/input/competitions/playground-series-s6e4/test.csv"

    OUTPUT_DIR = f"/kaggle/working/{EXP_ID}"

    TARGET_MAP = {"Low": 0, "Medium": 1, "High": 2}
    INV_TARGET_MAP = {0: "Low", 1: "Medium", 2: "High"}

def ensure_dir(path):
    os.makedirs(path, exist_ok=True)

def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def apply_multipliers(proba, mult):
    adjusted = proba * np.array(mult, dtype=np.float32).reshape(1, -1)
    return adjusted

def score_with_mult(proba, y_true, mult):
    adjusted = apply_multipliers(proba, mult)
    pred = adjusted.argmax(axis=1)
    return balanced_accuracy_score(y_true, pred)

ensure_dir(CFG.OUTPUT_DIR)

train = pd.read_csv(CFG.TRAIN_PATH)
test = pd.read_csv(CFG.TEST_PATH)
y = train[CFG.TARGET].map(CFG.TARGET_MAP).values

oof = np.load(CFG.OOF_PATH)
pred = np.load(CFG.PRED_PATH)

raw_cv = balanced_accuracy_score(y, oof.argmax(axis=1))

# coarse-to-fine search
best_mult = (1.0, 1.0, 1.0)
best_score = raw_cv

search_spaces = [
    np.arange(0.6, 1.41, 0.1),
    np.arange(-0.10, 0.101, 0.02),
    np.arange(-0.05, 0.051, 0.01),
]

# stage1: coarse absolute multipliers
for m0, m1, m2 in itertools.product(search_spaces[0], repeat=3):
    sc = score_with_mult(oof, y, (m0, m1, m2))
    if sc > best_score:
        best_score = sc
        best_mult = (m0, m1, m2)

# stage2-3: local refinement around best
for delta_grid in search_spaces[1:]:
    cur = np.array(best_mult, dtype=np.float32)
    candidates0 = cur[0] + delta_grid
    candidates1 = cur[1] + delta_grid
    candidates2 = cur[2] + delta_grid
    for m0, m1, m2 in itertools.product(candidates0, candidates1, candidates2):
        if min(m0, m1, m2) <= 0:
            continue
        sc = score_with_mult(oof, y, (m0, m1, m2))
        if sc > best_score:
            best_score = sc
            best_mult = (float(m0), float(m1), float(m2))

opt_oof = apply_multipliers(oof, best_mult)
opt_pred = apply_multipliers(pred, best_mult)

opt_cv = balanced_accuracy_score(y, opt_oof.argmax(axis=1))

submission = pd.DataFrame({
    CFG.ID_COL: test[CFG.ID_COL],
    CFG.TARGET: pd.Series(opt_pred.argmax(axis=1)).map(CFG.INV_TARGET_MAP)
})
submission.to_csv(f"{CFG.OUTPUT_DIR}/submission_dao_xgb_pair_te_posthoc_only.csv", index=False)

np.save(f"{CFG.OUTPUT_DIR}/oof_dao_xgb_pair_te_posthoc_only_proba.npy", opt_oof)
np.save(f"{CFG.OUTPUT_DIR}/pred_dao_xgb_pair_te_posthoc_only_proba.npy", opt_pred)
np.save(f"{CFG.OUTPUT_DIR}/best_class_multipliers.npy", np.array(best_mult, dtype=np.float32))

cv_result = {
    "exp_id": CFG.EXP_ID,
    "raw_cv": float(raw_cv),
    "optimized_cv": float(opt_cv),
    "best_multipliers": list(best_mult),
    "notes": {
        "base_parent": "exp_20260410_034_dao_xgb_pair_te_core_min",
        "pseudo_label": False,
        "formula_logit": False,
        "posthoc_optimization": True,
        "optimization_type": "class_multiplier_search"
    }
}
save_json(cv_result, f"{CFG.OUTPUT_DIR}/cv_result.json")

print("raw_cv:", raw_cv)
print("optimized_cv:", opt_cv)
print("best_multipliers:", best_mult)

raw_cv: 0.9773551303236498
optimized_cv: 0.979023960478168
best_multipliers: (0.5799999642372131, 0.51, 1.54)
